# 01 — The Agent Loop

> *"Every major AI agent runs the same core loop. A `while` loop that calls an LLM, checks if the response contains tool calls, executes them if it does, and stops if it doesn't."*

**By the end of this notebook you will:**
1. Write the 6-line agent loop from memory
2. Watch what really happens — including the **cognitive-shortcut** behavior where the model ignores your tools and answers from training
3. See the loop actually do useful multi-step work when forced
4. Add the four production necessities: bounded steps, parallel tool calls, error isolation, observability

Lecture reference: **S8 §3** (*Implementing the Agent Loop*).


## Setup

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
client = AsyncOpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Using model: {MODEL}")

## 1. The 6-line agent loop

This is the entire idea of an LLM agent in 6 lines of meaningful code:

```
while not done:
    msg = llm(messages, tools=tools)        # 1. ask
    messages.append(msg)                    # 2. remember
    if not msg.tool_calls: return msg.text  # 3. natural exit
    for tc in msg.tool_calls:               # 4. act
        result = execute(tc)                # 5. observe
        messages.append(result)             # 6. remember the observation
```

Below is the working version. The **core 6 lines** are still in there — everything else (the `print()` calls, the `max_steps` cap) is for visibility, not the algorithm. Watching the prompt evolve at every step is the whole point of this notebook.

In [ ]:
async def run_agent_v0(user_msg, tools, executors, max_steps=10):
    """The 6-line agent loop with verbose tracing. Break it on purpose to learn."""
    messages = [{"role": "user", "content": user_msg}]

    for i in range(max_steps):
        print(f"\n{'='*20} STEP {i+1} PROMPT {'='*20}")
        for m in messages:
            role = m.get("role")
            content = m.get("content") or (m.get("tool_calls") if "tool_calls" in m else "")
            print(f"[{role.upper()}]: {content}")

        resp = await client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))

        if not msg.tool_calls:
            print(f"\n[FINAL RESPONSE]: {msg.content}")
            return msg.content

        for tc in msg.tool_calls:
            func_name = tc.function.name
            args = tc.function.arguments
            print(f"\n[TOOL CALL]: Calling {func_name} with {args}")

            result = executors[func_name](**json.loads(args))

            print(f"[TOOL RESULT]: {result}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    raise RuntimeError("max_steps exceeded")

## 2. Try it with one tool

Single tool: `add`. The model has to *decide* to call this tool to compute `17 + 25` rather than computing it itself in text.

Schema note: `additionalProperties: False` is required when `strict: True` is on — we cover strict mode in detail in notebook 02.

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "add",
        "description": "Add two integers and return the sum.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}]

EXECUTORS = {"add": lambda a, b: {"sum": a + b}}

answer = await run_agent_v0("What is 17 + 25?", TOOLS, EXECUTORS)

Two steps. Step 1 — model calls `add(17, 25)` → 42. Step 2 — model has the tool result, produces the final answer.

Read the **STEP 2 PROMPT** carefully. Notice what's in the messages list:
1. The original user question
2. The assistant message — content is empty, `tool_calls` field has the `add(17, 25)` call
3. The tool result — `{"sum": 42}`

**That's how ReAct works.** The model sees its own prior decisions AND the tool observations on every iteration — that's how it can reason about whether it's done.


## 3. Break it — and watch the model improvise

Now the interesting test. Ask the agent to **multiply**. We only registered `add`. There's no `multiply` tool.

Before you run it, write down what you predict will happen. Pick one:
- (a) The model hallucinates a `multiply` tool name → `KeyError` crashes the agent
- (b) The model loops with `add` to compute 7+7+7+7+7+7+7+7 = 56 (full grounded loop)
- (c) The model calls `add(7, 8)` once, sees 15, then declares 56 from memory
- (d) The model ignores your tool entirely and just answers from training data
- (e) The model loops with `add` partially (a few times), then jumps to 56 from memory

Now run it. Then run it again. Then run it a third time.

In [ ]:
answer = await run_agent_v0("What is 7 times 8?", TOOLS, EXECUTORS)

Run that cell **at least three times** before reading on. Watch how the trace differs across runs. **Same model, same prompt, same tools** — different behavior every time.

These are real `gpt-4o-mini` 2026 traces we've observed for this exact prompt:

**Trace shape A** — *partial loop + late bypass* (the most common):
```
step 1: add(7, 7) = 14          ← starts repeated-addition reasoning
step 2: add(14, 7) = 21
step 3: add(21, 7) = 28
step 4: FINAL — "7 times 8 equals 56."   ← jumps from 28 (= 4×7) to 56 from memory
```

**Trace shape B** — *immediate bypass*:
```
step 1: add(7, 8) = 15          ← picks the closest tool, gets the wrong domain answer
step 2: FINAL — "That's addition, not multiplication. 7 × 8 = 56."
```

**Trace shape C** — *full grounded loop* (rare on `gpt-4o-mini`, more common on `gpt-4o`):
```
step 1-7: add(7, 7), add(14, 7), add(21, 7), add(28, 7), add(35, 7), add(42, 7), add(49, 7)
step 8: FINAL — "56."
```

**Two lessons stacked here. Both are senior-level.**


### Lesson 1 — Agents are non-deterministic

Same input → different traces. `temperature=1.0` (the default) means the model samples; tool choice is a sample; even the number of loop iterations is a sample. **You cannot reason about agent behavior from a single trace.**

Production teams test with N runs (typically 10-50) against a fixed eval set and report **distributions**: "the agent calls the right tool 87% of the time" — not "the agent calls the right tool" (a meaningless statement). This is why eval frameworks (Langfuse, Braintrust, LangSmith) all default to multi-run sampling.

**Interview signal:** if someone shows you one happy-path trace and concludes "the agent works," they haven't shipped agents in production.

### Lesson 2 — Partial loop + late bypass is the dangerous failure mode

Trace shape A — the *partial-loop* one — is what you should be afraid of.

The model **used the tool a few times**, then **jumped to a memory answer for the final step**. The trace looks like the agent was using your tool. The final answer was NOT grounded in the tool. The user reading the answer can't tell the difference.

Why this is dangerous:
- **Trace inspection is fooled** — "I see 3 tool calls in the log, so the agent used tools." But the *final answer* didn't use the tool result.
- **Grounding guarantees are broken** — if the whole point of the tool was real-time data (stock price, inventory count, DB lookup), the answer is *not* grounded even though tools were called.
- **Hard to detect at scale** — the answer might be right, might be wrong; you only catch it on the wrong ones, and you only catch wrong ones via downstream complaints.

**Senior interview answer:** *"The classic agent failure mode in 2026 isn't hallucinated tool names — modern models rarely do that. It's the **partial-grounding** problem: the model calls tools for a few steps, then synthesizes a final answer that bypasses the latest tool results. Trace counts are not grounding proofs. To enforce grounding you need either output validators that check the answer is derivable from the captured tool results, or structured outputs that reference tool call IDs by construction."*

**The 2023→2026 narrative shift:**

| What 2023 tutorials said | What 2026 models actually do |
|---|---|
| LLMs hallucinate tool names | Modern LLMs (`gpt-4o-mini`+) rarely do; they pick *closest* tool instead |
| The ReAct loop forces tool use | The loop forces nothing — the model can answer from training at any step |
| You can read the trace to verify the agent used tools | You need to verify the *final answer* came from tool data, not just that tools were *called* |
| Single happy-path trace = it works | N-run distributions across an eval set = it (probably) works |

**The implication for engineers**: run your agent 10 times against representative inputs. Inspect the **distribution** of trace shapes. The boring ones are fine — the failure-mode ones tell you what to defend against.

## 4. Force the loop — when the model really *has* to call the tool

If you want to see the loop genuinely doing multi-step work, you need a problem the model **can't shortcut from memory**. Big numbers work — `gpt-4o-mini` doesn't reliably add 5-digit numbers from training, so it defers to the tool.

Try this:

In [ ]:
answer = await run_agent_v0(
    "Use the add tool to compute the running total of these numbers, one at a time: "
    "12345, 67890, 11111, 22222. Report each intermediate total before the final.",
    TOOLS, EXECUTORS,
)

Read the trace top-to-bottom. You should see:

- **Step 1** — model calls `add(12345, 67890)` → `{"sum": 80235}`
- **Step 2** — model calls `add(80235, 11111)` → `{"sum": 91346}`
- **Step 3** — model calls `add(91346, 22222)` → `{"sum": 113568}`
- **Step 4** — final response with all four running totals

**This is ReAct working as advertised.** Each step the model:
1. Reads the previous tool result from the conversation history
2. Decides what to do next based on it
3. Either calls another tool or stops

Compare this trace with §3 above. Same loop. Same model. Same tool. Completely different behavior — driven entirely by whether the prompt forces tool use.

**The lesson is the contrast, not either trace alone.** In §3 the model bypassed the tool. In §4 it didn't. Production engineers run **both** kinds of trace before claiming an agent works.

## 5. What's missing for production

`run_agent_v0` is fine for learning. It's not safe for production. Four gaps:

| Gap | What it costs you |
|---|---|
| No `parallel_tool_calls=True` | independent tool calls happen serially — extra round-trip per call |
| No `try/except` around tool execution | unknown tool name or args error = **agent crashes**, user sees 500 |
| `print()` is your only trace | great for notebooks, useless for ops at scale |
| `raise RuntimeError` on max_steps | user gets a 500 instead of "here's my best answer so far" |

Let's fix all four.

In [ ]:
async def run_agent(user_msg, tools, executors, system=None, max_steps=10, model=None):
    model = model or MODEL
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})
    trace = []

    for step in range(max_steps):
        resp = await client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            parallel_tool_calls=True,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))
        trace.append({
            "step": step,
            "tokens": resp.usage.total_tokens,
            "tool_calls": [tc.function.name for tc in (msg.tool_calls or [])],
            "content": msg.content,
        })

        # Natural exit — no more tool calls
        if not msg.tool_calls:
            return {"answer": msg.content, "steps": step + 1, "trace": trace, "terminated_by": "natural"}

        # Execute tool calls — convert any error into an observation the LLM can read
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
                fn = executors.get(tc.function.name)
                if fn is None:
                    raise KeyError(f"Unknown tool: {tc.function.name}. Available: {list(executors)}")
                result = await fn(**args) if asyncio.iscoroutinefunction(fn) else fn(**args)
                content = json.dumps({"success": True, "data": result})
            except Exception as e:
                content = json.dumps({"success": False, "error": str(e)})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})

    # Max steps hit — force a final text answer instead of raising
    messages.append({
        "role": "user",
        "content": "Max steps reached. Provide your best answer based on what you've learned. Do not call any more tools.",
    })
    final = await client.chat.completions.create(model=model, messages=messages, tool_choice="none")
    return {
        "answer": final.choices[0].message.content,
        "steps": max_steps,
        "trace": trace,
        "terminated_by": "max_steps",
    }

## 6. Production v1 in action

Run the same forced-loop case from §4 with the production loop. Now we get a **structured trace** (no print spam) and **parallel-tool-call safety**.

In [ ]:
result = await run_agent(
    "Compute the sum of these numbers using add — one call at a time: 12345, 67890, 11111, 22222.",
    TOOLS,
    EXECUTORS,
    system="You are a calculator. Always use the add tool — do not compute in your head.",
)
print("answer:", result["answer"])
print("\ntrace:")
for entry in result["trace"]:
    calls = entry['tool_calls'] or '—'
    print(f"  step {entry['step']}: tools={calls}  tokens={entry['tokens']}")

## 7. Parallel tool calls

When the LLM has *independent* sub-questions it can issue multiple tool calls in a single turn. `parallel_tool_calls=True` lets that happen. Without it you'd get N round-trips for N independent calls.

Watch the `tool_calls` column in the trace below — step 0 should show three calls.

In [ ]:
result = await run_agent(
    "Compute 1+1, 2+2, and 3+3 — give me all three answers.",
    TOOLS,
    EXECUTORS,
)
print("answer:", result["answer"])
print("\ntrace:")
for entry in result["trace"]:
    print(f"  step {entry['step']}: {entry['tool_calls']}")

Step 0 should show `['add', 'add', 'add']` — three tool calls in one turn. Step 1 is the final answer.

Without `parallel_tool_calls=True` you'd see three separate steps each with one call — three extra round-trips. On a 25-step research task this can save 30-40% on cost.


## 8. Self-check

1. Write `run_agent_v0` from memory. (Six core lines, ignore the print scaffolding.)
2. Name three ways a model might NOT use a tool you registered. (Hint: §3 shows three trace shapes.)
3. What does `tool_choice="none"` guarantee about the next assistant message? When do we use it in `run_agent`?
4. Why do we wrap tool execution in `try/except` and convert errors to observations — instead of letting them raise?
5. In §3, the model bypassed the tool. In §4, it didn't. What was different about the prompt? Generalise the rule.
6. **Partial-loop + late-bypass** — explain why this trace shape is more dangerous than full bypass. How would you detect it programmatically?
7. **Non-determinism** — you run the same prompt twice and get different answers. Is the agent broken? What do you measure instead of "does it work"?

## What's next

Notebook **02** turns the loose `TOOLS` and `EXECUTORS` dicts into proper `BaseTool` and `ToolRegistry` classes — and digs into OpenAI's `strict: True` mode (the production must-have for preventing argument-shape hallucinations).
